# Import libraries

In [1]:
import pandas as pd
import time
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

True

# LLMs

In [2]:
class DefaultResponse(BaseModel):
    response: str

In [3]:
class Bielik:
    def __init__(self):
        self.__base_url = "https://153.19.237.85/api/llm/prompt/chat"
        auth = (os.getenv("LLM_USERNAME"), os.getenv("LLM_PASSWORD"))
        self.__auth_kwargs = {
            'auth': auth,
            'verify': False, # Disable SSL verification
        }

    def generate_response(self, system_prompt, user_prompt):
        data = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "max_length": 8192,  # adjust as needed
            "temperature": 0.0,
        }

        response = requests.put(
            self.__base_url,
            json=data,
            headers={
                'Accept': 'application/json',
                'Content-Type': 'application/json'
            },
            **self.__auth_kwargs,
        )
        response.raise_for_status()
        response_json = response.json()

        return response_json

In [4]:
class GPT:
    def __init__(self):
        self.__client = OpenAI()

    def generate_response(self, system_prompt, user_prompt, output_schema=DefaultResponse, model_name="gpt-4.1"):
        response = self.__client.responses.parse(
            model=model_name,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            text_format=output_schema,
        )

        return response.output_parsed


# Prompts

In [5]:
class FactCheck(BaseModel):
    statement: str = Field(
        description="Pojedyncze, wyizolowane zdanie lub stwierdzenie wyciągnięte z uzasadnienia."
    )
    is_true: bool = Field(
        description="Czy to konkretne stwierdzenie jest w 100% zgodne z prawdą historyczną, naukową lub logiczną?"
    )
    error_explanation: str | None = Field(
        description="Jeśli is_true to False, napisz krótko, na czym polega błąd. Jeśli True, zostaw null/puste."
    )

class EvaluationResponse(BaseModel):
    extracted_facts: list[FactCheck] = Field(
        description="Rozbij całe uzasadnienie na pojedyncze, atomowe fakty i oceń prawdziwość każdego z nich z osobna."
    )
    is_answer_correct: bool = Field(
        description=(
            "Zwróć True TYLKO wtedy, gdy oceniana odpowiedź końcowa jest merytorycznie "
            "w 100% zgodna z wzorcem. Zwróć False, jeśli występuje jakiekolwiek odstępstwo, "
            "błąd lub jeśli odpowiedź jest tylko częściowo poprawna."
        )
    )
    is_reasoning_correct: bool = Field(
        description=(
            "Zwróć True TYLKO ORAZ WYŁĄCZNIE, jeśli uzasadnienie jest absolutnie bezbłędne "
            "zarówno pod kątem logicznym, jak i FAKTOGRAFICZNYM. Wszystkie użyte w nim daty, liczby, "
            "nazwy i fakty muszą być w 100% prawdziwe. "
            "Zwróć BEZWZGLĘDNIE False, jeśli w uzasadnieniu pojawi się JAKIKOLWIEK błąd rzeczowy "
            "(np. błędny rok), fałszywe założenie, halucynacja lub zgadywanie, nawet jeśli "
            "końcowy wynik jest poprawny. "
            "Jedyny wyjątek, kiedy zwracasz True pomimo błędu, to omyłka polegająca na złym przepisaniu "
            "poprawnego wyniku z uzasadnienia do ostatecznej odpowiedzi."
        )
    )

calibrated_reasoning_prompt = """
Jesteś niezwykle surowym ewaluatorem faktów i logiki.
Twoim zadaniem jest ocena uzasadnienia i odpowiedzi końcowej.

ZASADA ZERO TOLERANCJI: Wymagam absolutnej precyzji. Najpierw musisz rozbić uzasadnienie na pojedyncze fakty i zweryfikować każdy z nich. Wystąpienie choćby jednego błędnego faktu (np. fałszywa data, zła liczba) bezwzględnie dyskwalifikuje całe uzasadnienie.

### DANE WEJŚCIOWE:
Pytanie: {question}
Wzorzec (Poprawna odpowiedź): {correct_answer}
Wygenerowane uzasadnienie: {reason}

### ZADANIE:
Wypełnij schemat. Pamiętaj: najpierw rozbij tekst na fakty i bezlitośnie je zweryfikuj. Jeśli jakikolwiek wyciągnięty fakt jest błędny, główna flaga 'is_reasoning_correct' musi wynosić False.
"""

In [6]:
medical_system_prompt = f"""
Jesteś wybitnym lekarzem i ekspertem medycznym o encyklopedycznej wiedzy.
Twoim zadaniem jest rozwiązywanie pytań testowych krok po kroku. Twój proces myślowy musi być precyzyjny: przeanalizuj każdą dostępną opcję przed wydaniem ostatecznego werdyktu.
"""

medical_user_prompt = """
Rozwiąż poniższe pytanie medyczne, opierając się na dostarczonych informacjach.

### DODATKOWE DANE / OPIS PRZYPADKU:
{data}

### TREŚĆ PYTANIA I DOSTĘPNE OPCJE:
{question}
"""

# Define metrics

In [7]:
def measure_response_time(function, *args, **kwargs):
    start = time.perf_counter()
    result = function(*args, **kwargs)
    end = time.perf_counter()

    return end - start, result

In [8]:
def count_words(text):
    text = text.split()

    return len(text)

In [9]:
def calibrated_reasoning(question: str, correct_answer: str, reason: str) -> EvaluationResponse:
    gpt = GPT()

    prompt = calibrated_reasoning_prompt.format(
        question=question,
        correct_answer=correct_answer,
        reason=reason,
    )

    response = gpt.generate_response(prompt, "", EvaluationResponse, "gpt-5.1")
    return response

# Generate answers

In [10]:
data = pd.read_json("data/lek_pl_sample.json")
data.drop(["edition", "year", "season", "question_id"], axis=1, inplace=True)
data.reset_index(inplace=True, drop=True)
data.rename(columns={"question_w_options": "question"}, inplace=True)

OUTPUT_FILE = "data/results.jsonl"

def save_result(record, file_path=OUTPUT_FILE):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


model = GPT()

for i in range(237, len(data)):
    question = data.iloc[i]["question"]
    correct_answer = data.iloc[i]["answer"]

    response_time, response = measure_response_time(
        model.generate_response,
        medical_system_prompt,
        medical_user_prompt.format(question=question, data="")
    )

    text = response.response
    used_words = count_words(text)

    score = calibrated_reasoning(question, correct_answer, text)

    print(f"Pytanie: {i}")
    print(text)
    print(score)

    is_answer_correct = float(score.is_answer_correct)
    is_reasoning_correct = float(score.is_reasoning_correct)

    facts = score.extracted_facts
    fact_accuracy = sum(f.is_true for f in facts) / len(facts) if facts else 0.0

    all_facts = [
        {
            "statement": f.statement,
            "is_true": f.is_true,
            "error_explanation": f.error_explanation
        }
        for f in score.extracted_facts
    ]

    record = {
        "question": question,
        "correct_answer": correct_answer,
        "response": text,
        "response_time": response_time,
        "used_words": used_words,
        "is_answer_correct": is_answer_correct,
        "is_reasoning_correct": is_reasoning_correct,
        "fact_accuracy": fact_accuracy,
        "all_facts": all_facts,
    }

    save_result(record)



Pytanie: 237
Przeanalizujmy każdą opcję krok po kroku:

**A. Niewinny szmer skurczowy**
- Niewinny szmer skurczowy zwykle nie wiąże się z objawami (np. bólem w klatce piersiowej), nie nasila się przy zmianie pozycji na stojącą, nie towarzyszą mu zmiany w EKG ani dodatni wywiad rodzinny w kierunku nagłych zgonów sercowych. Ta opcja jest mało prawdopodobna.

**B. Stenoza aortalna**
- Typowo daje szmer skurczowy, ale najgłośniejszy jest w punkcie Erba lub nad aortą, promieniuje do tętnic szyjnych, a nie wzdłuż lewego brzegu mostka. Objawy pojawiają się zwykle w starszym wieku. Szmer nie nasila się przy wstawaniu. Mało prawdopodobne.

**C. Kardiomiopatia przerostowa (HCM)**
- Typowo występuje u młodych dorosłych, często z dodatnim wywiadem rodzinnym nagłych zgonów sercowych. Szmer skurczowy słyszalny wzdłuż lewego brzegu mostka, który nasila się przy zmianie pozycji na stojącą (zmniejszenie powrotu żylnego zwiększa gradient w drodze odpływu lewej komory). Objawy mogą obejmować ból w klatce

In [13]:
dat = pd.read_json("data/results.jsonl", lines=True)

In [14]:
dat

,question,correct_answer,response,response_time,used_words,is_answer_correct,is_reasoning_correct,fact_accuracy,all_facts
0,Do nowotworów germinalnych jajnika nie należy:...,D,"Aby rozwiązać to pytanie, przeanalizujmy każdą...",4.305826,104,1,1,1.0,[{'statement': 'Nowotwory germinalne jajnika t...
1,Podstawowym lekiem stosowanym w POChP są gliko...,E,Przeanalizujmy każde zdanie:\n\n**Zdanie 1:** ...,3.339572,131,1,1,1.0,[{'statement': 'Podstawowymi lekami w POChP są...
2,Niewydolność cieśniowo-szyjkowa w wywiadzie je...,A,Przeanalizujmy każdą z opcji:\n\n**A. założeni...,5.936378,196,1,1,1.0,[{'statement': 'Profilaktyczny szew okrężny na...
3,Nazwę „schizofrenia” wprowadził:\nA. Kraepelin...,E,Przeanalizujmy każdą z opcji:\n\nA. Kraepelin ...,2.866578,103,1,1,1.0,[{'statement': 'Emil Kraepelin był niemieckim ...
4,W ciągu ostatniego roku zakażenie Legionella p...,A,Przeanalizujmy każdą z odpowiedzi:\n\n**A. u m...,4.792016,176,1,1,1.0,[{'statement': 'Legionelloza (choroba legionis...
...,...,...,...,...,...,...,...,...,...
395,Zgodnie z aktualnie obowiązującym Programem Sz...,C,Przeanalizujmy każdą z opcji w kontekście Prog...,5.256586,144,0,0,0.5,[{'statement': 'Osoby wykonujące zawody medycz...
396,Prawidłowy wynik konwencjonalnej radiografii k...,A,Przeanalizujmy każdą część pytania:\n\n**Stwie...,7.983490,172,1,1,1.0,[{'statement': 'Prawidłowy wynik konwencjonaln...
397,Prawidłowy kardiotokograficzny test niestresow...,C,Przeanalizujmy każdą z opcji:\n\n**Prawidłowy ...,4.812038,132,1,1,1.0,[{'statement': 'Prawidłowa podstawowa czynność...
398,"Wskaż, kiedy lekarz onkolog może ograniczyć in...",E,Przeanalizujmy każdą z opcji:\n\n**A. w sytuac...,4.089207,199,1,1,1.0,"[{'statement': 'Nie ma podstawy prawnej, która..."


In [12]:
import json
import pandas as pd

rows = []
with open("data/results.jsonl") as f:
    for line in f:
        try:
            rows.append(json.loads(line))
        except:
            print(line)
            input()# skip bad lines

df = pd.DataFrame(rows)

{"question": "Wskaż fałszywe stwierdzenie dotyczące stosowania pętlowych leków moczopędnych u pacjentów z nadciśnieniem tętniczym w przewlekłej chorobie nerek:\nA. pętlowe leki moczopędne są skuteczne nawet u chorych z filtracją kłębuszkową (GFR) < 15 ml/min/1,73 m2.\nB. stosowanie dużych dawek pętlowych leków moczopędnych może spowodować odwodnienie i zmniejszenie GFR.\nC. nie należy stosować pętlowych leków moczopędnych u chorych bez hiperkaliemii.\nD. stosowanie leków moczopędnych w połączeniu z inhibitorami konwertazy angiotensyny zmniejsza albuminurię.\nE. stosowanie dużych dawek pętlowych leków moczopędnych może spowodować uszkodzenie słuchu.", "correct_answer": "C", "response": "Przeanalizujmy każdą z odpowiedzi:\n\n**A. pętlowe leki moczopędne są skuteczne nawet u chorych z filtracją kłębuszkową (GFR) < 15 ml/min/1,73 m2.**\n- To prawda. Pętlowe leki moczopędne (np. furosemid) są skuteczne nawet przy bardzo niskiej filtracji kłębuszkowej, w przeciwieństwie do tiazydów.\n\n**B. 

KeyboardInterrupt: Interrupted by user